# 🛣️ Toll Maut Revenue & Traffic SQL Reporting Pipeline
### Multi-Table DuckDB Analytics with Automated Weekly Reports
**Author:** Gaurav Bhatia | MSc Data Science, GISMA University Berlin  
**Tools:** Python · DuckDB · Pandas · SQL (JOINs, CTEs, Window Functions) · Jupyter  
**GitHub:** gauravbhatia-bit  

---

## 📌 Project Overview

Toll operators like **Toll Collect** process millions of Mautdaten transactions daily. This project builds a complete SQL reporting pipeline using **DuckDB** to:
1. Design and populate a 4-table star schema
2. Write complex SQL (JOINs, CTEs, Window Functions)
3. Generate 4 automated business reports
4. Export structured CSV outputs

> ⚠️ *Data is synthetic and simulates German highway Mautdaten.*

## 1. 📦 Imports & Setup

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
con = duckdb.connect()
print('✅ DuckDB version:', duckdb.__version__)
print('✅ Connected to in-memory DuckDB')


## 2. 🗂️ Database Schema — Star Schema

```
toll_transactions (fact)
    ├── vehicle_id  →  vehicles (dim)
    ├── segment_id  →  segments (dim)
    └── date_id     →  calendar (dim)
```

| Table | Type | Description |
|---|---|---|
| `toll_transactions` | Fact | Every toll event: vehicle, segment, time, amount |
| `vehicles` | Dimension | Vehicle class, type, registration country |
| `segments` | Dimension | Highway name, location, length, road type |
| `calendar` | Dimension | Date attributes: week, month, holiday flag |

## 3. 🏗️ Create & Populate Dimension Tables

In [ ]:
vehicles_data = pd.DataFrame({
    'vehicle_id':       range(1, 21),
    'vehicle_class':    (['PKW']*8 + ['LKW_7.5t']*5 + ['LKW_18t']*4 + ['Bus']*3),
    'axle_count':       ([2]*8 + [2]*5 + [3]*4 + [3]*3),
    'registration':     (['DE']*10 + ['PL']*4 + ['NL']*3 + ['FR']*2 + ['CZ']*1),
    'emission_class':   (['Euro6']*8 + ['Euro5']*5 + ['Euro6']*4 + ['Euro5']*3),
    'toll_rate_per_km': ([0.0]*8 + [0.187]*5 + [0.274]*4 + [0.155]*3)
})
con.execute('CREATE TABLE vehicles AS SELECT * FROM vehicles_data')
print(f'✅ vehicles: {con.execute("SELECT COUNT(*) FROM vehicles").fetchone()[0]} rows')

segments_data = pd.DataFrame({
    'segment_id':   range(1, 6),
    'segment_name': ['A1_Nord','A9_Sued','A100_Berlin','A2_Ost','A7_West'],
    'start_city':   ['Hamburg','Munich','Berlin','Berlin','Hamburg'],
    'end_city':     ['Bremen','Nuremberg','Berlin','Warsaw','Hanover'],
    'length_km':    [45.2, 62.8, 11.5, 78.4, 53.1],
    'speed_limit':  [130, 130, 100, 130, 130],
    'road_type':    ['Autobahn','Autobahn','Stadtautobahn','Autobahn','Autobahn'],
    'toll_zone':    ['Nord','Sued','Berlin','Ost','West']
})
con.execute('CREATE TABLE segments AS SELECT * FROM segments_data')
print(f'✅ segments: {con.execute("SELECT COUNT(*) FROM segments").fetchone()[0]} rows')

dates = pd.date_range('2024-01-01', '2024-06-30')
german_holidays = ['2024-01-01','2024-03-29','2024-04-01','2024-05-01','2024-05-09','2024-05-20']
calendar_data = pd.DataFrame({
    'date_id':      range(1, len(dates)+1),
    'date':         dates,
    'year':         dates.year,
    'month':        dates.month,
    'month_name':   dates.strftime('%B'),
    'week_number':  dates.isocalendar().week.values,
    'quarter':      dates.quarter,
    'day_of_week':  dates.dayofweek,
    'day_name':     dates.strftime('%A'),
    'is_weekend':   (dates.dayofweek >= 5).astype(int),
    'is_holiday':   dates.strftime('%Y-%m-%d').isin(german_holidays).astype(int)
})
con.execute('CREATE TABLE calendar AS SELECT * FROM calendar_data')
print(f'✅ calendar: {con.execute("SELECT COUNT(*) FROM calendar").fetchone()[0]} rows')


## 4. 🏗️ Populate Fact Table — toll_transactions

In [ ]:
records = []
transaction_id = 1

for day_offset in range(182):
    date = datetime(2024, 1, 1) + timedelta(days=day_offset)
    is_weekend = date.weekday() >= 5
    for segment_id in range(1, 6):
        seg = segments_data.iloc[segment_id - 1]
        for hour in range(24):
            if 7 <= hour <= 9 or 17 <= hour <= 19:
                n = np.random.randint(40, 80)
            elif 22 <= hour or hour <= 5:
                n = np.random.randint(5, 20)
            else:
                n = np.random.randint(20, 45)
            if is_weekend:
                n = int(n * 0.6)
            for _ in range(n):
                vid = np.random.randint(1, 21)
                veh = vehicles_data.iloc[vid - 1]
                toll = round(veh['toll_rate_per_km'] * seg['length_km'] * np.random.normal(1.0, 0.05), 4)
                records.append({
                    'transaction_id': transaction_id,
                    'vehicle_id':     vid,
                    'segment_id':     segment_id,
                    'date_id':        day_offset + 1,
                    'hour':           hour,
                    'toll_amount':    max(toll, 0),
                    'direction':      np.random.choice(['North','South']),
                    'entry_time':     date.replace(hour=hour, minute=np.random.randint(0,59))
                })
                transaction_id += 1

transactions_df = pd.DataFrame(records)
con.execute('CREATE TABLE toll_transactions AS SELECT * FROM transactions_df')
count   = con.execute('SELECT COUNT(*) FROM toll_transactions').fetchone()[0]
revenue = con.execute('SELECT ROUND(SUM(toll_amount),2) FROM toll_transactions').fetchone()[0]
print(f'✅ toll_transactions: {count:,} rows')
print(f'💰 Total simulated revenue: €{revenue:,.2f}')


## 5. 🔍 Schema Verification & 4-Table JOIN Test

In [ ]:
tables = con.execute('SHOW TABLES').fetchdf()
print('Tables in database:')
print(tables.to_string(index=False))
print()

join_sql = (
    'SELECT t.transaction_id, v.vehicle_class, v.registration,'
    ' s.segment_name, s.road_type, c.date, c.day_name,'
    ' c.is_holiday, t.hour, t.toll_amount'
    ' FROM toll_transactions t'
    ' JOIN vehicles v ON t.vehicle_id = v.vehicle_id'
    ' JOIN segments s ON t.segment_id = s.segment_id'
    ' JOIN calendar c ON t.date_id    = c.date_id'
    ' LIMIT 5'
)
join_test = con.execute(join_sql).fetchdf()
print('4-table JOIN test (first 5 rows):')
print(join_test.to_string(index=False))


## 6. 📊 Report 1 — Weekly Revenue by Segment & Vehicle Class

**Business question:** How much toll revenue did each highway segment generate per week, by vehicle class?

SQL: `JOIN × 3`, `GROUP BY`, `SUM`, `ROUND`, `ORDER BY`

In [ ]:
r1 = con.execute(
    'SELECT c.week_number AS week, c.month_name AS month,'
    ' s.segment_name AS segment, s.toll_zone AS zone,'
    ' v.vehicle_class,'
    ' COUNT(t.transaction_id) AS total_transactions,'
    ' ROUND(SUM(t.toll_amount), 2) AS total_revenue_eur,'
    ' ROUND(AVG(t.toll_amount), 4) AS avg_toll_eur'
    ' FROM toll_transactions t'
    ' JOIN vehicles v ON t.vehicle_id = v.vehicle_id'
    ' JOIN segments s ON t.segment_id = s.segment_id'
    ' JOIN calendar c ON t.date_id    = c.date_id'
    ' GROUP BY c.week_number, c.month_name, s.segment_name, s.toll_zone, v.vehicle_class'
    ' ORDER BY c.week_number, total_revenue_eur DESC'
).fetchdf()
r1.to_csv('report1_weekly_revenue.csv', index=False)
print(f'✅ report1_weekly_revenue.csv — {len(r1):,} rows')
print()
print('Top 10 by revenue:')
print(r1.nlargest(10,'total_revenue_eur')[['week','segment','vehicle_class','total_transactions','total_revenue_eur']].to_string(index=False))


## 7. 📊 Report 2 — Peak Hour Traffic Analysis

**Business question:** Which hours are busiest per segment and how does revenue concentrate across time periods?

SQL: `CTE`, `CASE WHEN`, `RANK() OVER (PARTITION BY)`

In [ ]:
r2 = con.execute(
    'WITH hourly AS (SELECT s.segment_name, t.hour,'
    ' COUNT(t.transaction_id) AS transactions,'
    ' ROUND(SUM(t.toll_amount),2) AS revenue,'
    ' CASE WHEN t.hour BETWEEN 7 AND 9 THEN \'Morning Rush\''
    ' WHEN t.hour BETWEEN 17 AND 19 THEN \'Evening Rush\''
    ' WHEN t.hour >= 22 OR t.hour <= 5 THEN \'Night\''
    ' ELSE \'Off-Peak\' END AS time_category'
    ' FROM toll_transactions t JOIN segments s ON t.segment_id=s.segment_id'
    ' GROUP BY s.segment_name, t.hour),'
    ' ranked AS (SELECT *, RANK() OVER (PARTITION BY segment_name ORDER BY transactions DESC) AS hour_rank FROM hourly)'
    ' SELECT segment_name, hour, time_category, transactions, revenue, hour_rank FROM ranked ORDER BY segment_name, hour'
).fetchdf()
r2.to_csv('report2_peak_hours.csv', index=False)
print(f'✅ report2_peak_hours.csv — {len(r2)} rows')
print()
print('Busiest hour per segment:')
print(r2[r2['hour_rank']==1][['segment_name','hour','time_category','transactions','revenue']].to_string(index=False))


## 8. 📊 Report 3 — Heavy Vehicle Share & Revenue Contribution

**Business question:** What % of traffic and revenue comes from heavy vehicles (LKW) per segment per month?

SQL: `CTE × 2`, `JOIN × 3`, `LIKE`, percentage division

In [ ]:
r3 = con.execute(
    'WITH base AS (SELECT c.month_name, c.month, s.segment_name,'
    ' CASE WHEN v.vehicle_class LIKE \'LKW%\' THEN \'Heavy\' ELSE \'Light/Other\' END AS vehicle_type,'
    ' COUNT(t.transaction_id) AS transactions, ROUND(SUM(t.toll_amount),2) AS revenue'
    ' FROM toll_transactions t'
    ' JOIN vehicles v ON t.vehicle_id=v.vehicle_id'
    ' JOIN segments s ON t.segment_id=s.segment_id'
    ' JOIN calendar c ON t.date_id=c.date_id'
    ' GROUP BY c.month_name, c.month, s.segment_name, vehicle_type),'
    ' totals AS (SELECT month_name, month, segment_name,'
    ' SUM(transactions) AS total_tx, SUM(revenue) AS total_rev'
    ' FROM base GROUP BY month_name, month, segment_name)'
    ' SELECT b.month_name, b.segment_name, b.vehicle_type,'
    ' SUM(b.transactions) AS transactions,'
    ' ROUND(SUM(b.transactions)*100.0/t.total_tx,1) AS pct_of_traffic,'
    ' ROUND(SUM(b.revenue),2) AS revenue,'
    ' ROUND(SUM(b.revenue)*100.0/t.total_rev,1) AS pct_of_revenue'
    ' FROM base b JOIN totals t ON b.month_name=t.month_name AND b.segment_name=t.segment_name'
    ' GROUP BY b.month_name, b.month, b.segment_name, b.vehicle_type, t.total_tx, t.total_rev'
    ' ORDER BY b.month, b.segment_name, b.vehicle_type'
).fetchdf()
r3.to_csv('report3_heavy_vehicle_share.csv', index=False)
print(f'✅ report3_heavy_vehicle_share.csv — {len(r3)} rows')
print()
jan = r3[(r3['month_name']=='January') & (r3['vehicle_type']=='Heavy')]
print('Heavy vehicle revenue share — January:')
print(jan[['segment_name','pct_of_traffic','revenue','pct_of_revenue']].to_string(index=False))


## 9. 📊 Report 4 — Transaction Anomaly Flag Report

**Business question:** Which transactions deviate significantly from the segment average — flagging data errors or fraud?

SQL: `CTE`, `STDDEV`, `NULLIF`, Z-score calculation, `CASE WHEN` severity

In [ ]:
r4 = con.execute(
    'WITH stats AS (SELECT t.segment_id, s.segment_name, v.vehicle_class,'
    ' AVG(t.toll_amount) AS avg_toll, STDDEV(t.toll_amount) AS std_toll'
    ' FROM toll_transactions t'
    ' JOIN segments s ON t.segment_id=s.segment_id'
    ' JOIN vehicles v ON t.vehicle_id=v.vehicle_id'
    ' WHERE v.vehicle_class != \'PKW\''
    ' GROUP BY t.segment_id, s.segment_name, v.vehicle_class),'
    ' flagged AS (SELECT t.transaction_id, c.date, t.hour,'
    ' s.segment_name, v.vehicle_class, v.registration, t.toll_amount,'
    ' ss.avg_toll, ss.std_toll,'
    ' ROUND((t.toll_amount-ss.avg_toll)/NULLIF(ss.std_toll,0),2) AS z_score'
    ' FROM toll_transactions t'
    ' JOIN vehicles v ON t.vehicle_id=v.vehicle_id'
    ' JOIN segments s ON t.segment_id=s.segment_id'
    ' JOIN calendar c ON t.date_id=c.date_id'
    ' JOIN stats ss ON t.segment_id=ss.segment_id AND v.vehicle_class=ss.vehicle_class'
    ' WHERE v.vehicle_class != \'PKW\''
    ' AND ABS((t.toll_amount-ss.avg_toll)/NULLIF(ss.std_toll,0))>2.5)'
    ' SELECT *,'
    ' CASE WHEN ABS(z_score)>4 THEN \'CRITICAL\''
    ' WHEN ABS(z_score)>3 THEN \'HIGH\' ELSE \'MEDIUM\' END AS flag_severity'
    ' FROM flagged ORDER BY ABS(z_score) DESC LIMIT 100'
).fetchdf()
r4.to_csv('report4_anomaly_flags.csv', index=False)
print(f'✅ report4_anomaly_flags.csv — {len(r4)} rows')
print()
print('Top 10 flagged transactions:')
print(r4[['transaction_id','date','segment_name','vehicle_class','toll_amount','avg_toll','z_score','flag_severity']].head(10).to_string(index=False))


## 10. 📋 Executive KPI Summary

In [ ]:
summary = con.execute(
    'SELECT COUNT(DISTINCT t.transaction_id) AS total_transactions,'
    ' ROUND(SUM(t.toll_amount),2) AS total_revenue_eur,'
    ' ROUND(AVG(t.toll_amount),4) AS avg_toll_eur,'
    ' COUNT(DISTINCT t.segment_id) AS segments_monitored,'
    ' COUNT(DISTINCT t.vehicle_id) AS unique_vehicles,'
    ' COUNT(DISTINCT c.date) AS days_covered,'
    ' MIN(c.date) AS period_start, MAX(c.date) AS period_end'
    ' FROM toll_transactions t JOIN calendar c ON t.date_id=c.date_id'
).fetchdf()

print('=' * 52)
print('   TOLL MAUT REPORTING PIPELINE — KPI SUMMARY')
print('=' * 52)
for col in summary.columns:
    print(f'  {col:<28}: {summary[col].iloc[0]}')
print('=' * 52)
print()

rev_seg = con.execute(
    'SELECT s.segment_name, COUNT(*) AS transactions,'
    ' ROUND(SUM(t.toll_amount),2) AS total_revenue'
    ' FROM toll_transactions t JOIN segments s ON t.segment_id=s.segment_id'
    ' GROUP BY s.segment_name ORDER BY total_revenue DESC'
).fetchdf()
print('Revenue by Segment:')
print(rev_seg.to_string(index=False))


## 11. ✅ Pipeline Summary

| Report | Output File | SQL Techniques |
|---|---|---|
| Weekly Revenue | `report1_weekly_revenue.csv` | JOIN × 3, GROUP BY, SUM, ROUND |
| Peak Hour Traffic | `report2_peak_hours.csv` | CTE, CASE WHEN, RANK() window function |
| Heavy Vehicle Share | `report3_heavy_vehicle_share.csv` | CTE × 2, JOIN × 3, percentage division |
| Anomaly Flags | `report4_anomaly_flags.csv` | CTE, STDDEV, Z-score, NULLIF, CASE WHEN |

### Key Technical Highlights
- **Star schema** — 1 fact table + 3 dimension tables
- **4-way JOIN** across transactions, vehicles, segments, calendar
- **CTEs** for modular, readable multi-step queries
- **Window functions** — RANK() OVER (PARTITION BY) for per-segment ranking
- **Z-score anomaly detection** purely in SQL using STDDEV + NULLIF
- All 4 reports exported as structured **CSV** files

### Business Value
> This pipeline mirrors the exact Toll Collect Werkstudent workflow — analysis, reporting, monitoring and optimisation of Mautdaten using industry-standard SQL and DuckDB.

---
*Project by Gaurav Bhatia | gauravbhatia-bit | Berlin, 2026*